# PharmaFlow Foundation — Session 1.3 Solutions
### Working with APIs & JSON Data

These are the instructor solutions for all six exercise tiers. Every cell below has
been run for real — including every HTTP request — against the practice API server
this session ships with. The outputs shown are actual output, not illustrative text.

**Dataset and tools:** `practice_api_server.py` — a small local stand-in for the real
OpenFDA API — serving `openfda_practice_sample.json`, 300 real raw OpenFDA
adverse-event records, served 100 per page (3 pages; the live API allows up to 1,000), in
the exact response shape the live API uses. It deliberately rate-limits one request so
retry logic has something real to prove itself against. The same code shown here works
against the real `https://api.fda.gov/drug/event.json` with only the base URL changed
— which is exactly what the graded Lab asks you to do.


## Before You Run Anything — Environment Check

Same setup pattern as Sessions 1.1 and 1.2 — this notebook finds the repo
automatically, no matter which folder your notebook tool happened to start in.

Run the cell below first. If it prints an error instead of "You're ready to
go.", stop here and fix your setup before continuing.


In [1]:
# --- Setup check: always run this cell first ---
import os

def find_repo_root(marker_folder="data", start=None):
    """
    Same reasoning as every previous session: different notebook tools start
    a notebook's kernel in different folders, so we search upward from
    wherever we happened to start until we find a folder that contains
    `data` -- that only exists at the repo root, so wherever we find it
    IS the repo root.
    """
    current = os.path.abspath(start or os.getcwd())
    while True:
        if os.path.isdir(os.path.join(current, marker_folder)):
            return current
        parent = os.path.dirname(current)
        if parent == current:
            raise FileNotFoundError(
                f"Could not find a '{marker_folder}' folder anywhere above "
                f"{os.getcwd()}. Make sure this notebook is somewhere inside "
                f"the pharmaflow-ai repo."
            )
        current = parent


REPO_ROOT = find_repo_root()
SAMPLE_FILE = os.path.join(REPO_ROOT, "data", "raw", "openfda_practice_sample.json")

if os.path.exists(SAMPLE_FILE):
    print("Repo root found at:", REPO_ROOT)
    print("Practice sample found at:", SAMPLE_FILE)
    print("You're ready to go.")
else:
    print("Repo root found at:", REPO_ROOT)
    print("Missing file:", SAMPLE_FILE)
    print()
    print("Fix: confirm openfda_practice_sample.json is in data/raw/ at the")
    print("top level of the pharmaflow-ai repo.")


Repo root found at: /workspaces/pharmaflow-ai
Practice sample found at: /workspaces/pharmaflow-ai/data/raw/openfda_practice_sample.json
You're ready to go.


## Warm-Up — Two pages, one list (no real API yet)

Before any real API is involved, prove you can combine results that arrive in
separate pages into one list, and reach into a nested field -- on 3 invented
library records small enough to check by eye.


In [2]:
# WARM-UP: tiny invented data, no FDA/API records yet.
toy_page_1 = {"results": [
    {"book_id": "B1", "info": {"title": "Fox in Socks", "pages": 72}},
    {"book_id": "B2", "info": {"title": "Green Eggs and Ham", "pages": 62}},
]}
toy_page_2 = {"results": [
    {"book_id": "B3", "info": {"title": "The Cat in the Hat", "pages": 61}},
]}

all_books = toy_page_1["results"] + toy_page_2["results"]
titles = [book["info"]["title"] for book in all_books]

print(f"all_books has {len(all_books)} entries")
print("titles:", titles)


all_books has 3 entries
titles: ['Fox in Socks', 'Green Eggs and Ham', 'The Cat in the Hat']


## Exercise 1 · Beginner — One page, for real

Start the practice server, then make a genuine HTTP GET request for the
first page and confirm the pipeline can actually reach it.


In [5]:
# Exercise 1: start the practice server and fetch one real page.
import time
import requests
from practice_api_server import start_server

server, PRACTICE_URL = start_server()
time.sleep(0.3)  # give the background thread a moment to come up

PAGE_SIZE = 100


def fetch_page(skip, limit=PAGE_SIZE, base_url=PRACTICE_URL, max_retries=3):
    """Fetch one page of records from the API, retrying with a short
    backoff if the API responds 429 (rate limited)."""
    for attempt in range(max_retries):
        response = requests.get(base_url, params={"limit": limit, "skip": skip})
        if response.status_code == 200:
            return response.json().get("results", [])
        if response.status_code == 429:
            wait = 2 ** attempt
            print(f"  Rate limited at skip={skip}, retrying in {wait}s...")
            time.sleep(wait)
            continue
        response.raise_for_status()
    raise RuntimeError(f"Still rate-limited after {max_retries} retries (skip={skip}).")


page_0 = fetch_page(0)
print("records returned:", len(page_0))
print("first record safety_report_id:", page_0[0]["safetyreportid"])


records returned: 100
first record safety_report_id: 5801206-7


## Exercise 2 · Intermediate — All three pages

One page of 100 isn't enough -- paginate through all 300 records in the
practice sample.


In [6]:
# Exercise 2: paginate through every page until all records are collected.
def fetch_all_pages(total_records, page_size=PAGE_SIZE, base_url=PRACTICE_URL):
    """Paginate through the API until total_records have been collected
    or the API runs out of records."""
    all_results = []
    skip = 0
    while len(all_results) < total_records:
        page = fetch_page(skip, limit=page_size, base_url=base_url)
        if not page:
            break
        all_results.extend(page)
        skip += page_size
    return all_results[:total_records]


all_records = fetch_all_pages(300)
print("total records fetched:", len(all_records))
print("last record safety_report_id:", all_records[-1]["safetyreportid"])


  Rate limited at skip=200, retrying in 1s...
total records fetched: 300
last record safety_report_id: 10003602


## Exercise 3 · Advanced — Don't crash on 429

`fetch_page()` above already has the retry logic built in (that's what the
"Rate limited..." line above was proving) -- this cell makes it visible on
its own by re-running the full pagination fresh and timing it.


In [7]:
# Exercise 3: confirm the retry survives a fresh run and doesn't break anything.
import time as _time

server.seen_skips.clear()  # practice server: let skip=200 rate-limit us again
t0 = _time.time()
all_records_retry_proof = fetch_all_pages(300)
elapsed = _time.time() - t0

print(f"fetched {len(all_records_retry_proof)} records in {elapsed:.1f}s, surviving one 429")
assert len(all_records_retry_proof) == 300
print("confirmed: still 300 records after the retry fired")


  Rate limited at skip=200, retrying in 1s...
fetched 300 records in 1.1s, surviving one 429
confirmed: still 300 records after the retry fired


## Exercise 4 · Corporate Scenario — Flatten it, save it as CSV

Regulatory Affairs needs the same flat fields this pipeline has produced
since Session 1.2 -- not the deeply nested shape the API actually returns --
written to CSV instead of JSON this time.


In [8]:
# Exercise 4: flatten nested records and write the usable ones to CSV.
import csv


def flatten_record(raw_record):
    """Pull the flat fields this pipeline needs out of one raw, deeply
    nested OpenFDA record. Returns None if the record doesn't have exactly
    one suspect drug and a non-empty first reaction."""
    drugs = raw_record.get("patient", {}).get("drug", [])
    suspect_drugs = [
        d for d in drugs
        if d.get("drugcharacterization") == "1" and d.get("medicinalproduct")
    ]
    reactions = raw_record.get("patient", {}).get("reaction", [])
    if len(suspect_drugs) != 1:
        return None
    if not reactions or not reactions[0].get("reactionmeddrapt"):
        return None
    return {
        "safety_report_id": raw_record.get("safetyreportid"),
        "drug_name": suspect_drugs[0]["medicinalproduct"],
        "reaction": reactions[0]["reactionmeddrapt"],
        "serious": raw_record.get("serious"),
        "patient_age": raw_record.get("patient", {}).get("patientonsetage"),
        "country": raw_record.get("occurcountry"),
        "event_date": raw_record.get("receivedate"),
    }


def write_to_csv(flat_records, filepath):
    """Write a list of flat record dicts to CSV, one row per record, using
    the dict keys from the first record as the header."""
    if not flat_records:
        raise ValueError("Nothing to write -- flat_records is empty.")
    fieldnames = list(flat_records[0].keys())
    with open(filepath, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(flat_records)


flattened = [flatten_record(r) for r in all_records]
usable = [f for f in flattened if f is not None]
print(f"usable: {len(usable)} / {len(all_records)}")

age_known = sum(1 for u in usable if u["patient_age"])
country_known = sum(1 for u in usable if u["country"])
print(f"age known: {age_known} / {len(usable)}")
print(f"country known: {country_known} / {len(usable)}")

OUT_DIR = os.path.join(REPO_ROOT, "data", "processed")
os.makedirs(OUT_DIR, exist_ok=True)
OUT_FILE = os.path.join(OUT_DIR, "adverse_events_from_api.csv")
write_to_csv(usable, OUT_FILE)

with open(OUT_FILE, newline="") as f:
    rows = list(csv.DictReader(f))
print(f"wrote and re-read {len(rows)} rows")
print("first row:", rows[0])


usable: 191 / 300
age known: 149 / 191
country known: 188 / 191
wrote and re-read 191 rows
first row: {'safety_report_id': '5801206-7', 'drug_name': 'DURAGESIC-100', 'reaction': 'DRUG ADMINISTRATION ERROR', 'serious': '1', 'patient_age': '26', 'country': '', 'event_date': '20080707'}


## Exercise 5 · Interview Challenge — Never crash, whatever shape

A standalone, hardened usability check that must never raise, no matter what
shape of record it's handed.


In [9]:
# Exercise 5: a pure, defensive check -- never raises.
def is_usable_record(raw_record):
    """Return True if raw_record has exactly one suspect drug and a
    non-empty first reaction. Never raises, whatever shape raw_record is."""
    try:
        drugs = raw_record.get("patient", {}).get("drug", [])
        reactions = raw_record.get("patient", {}).get("reaction", [])
        suspect = [
            d for d in drugs
            if d.get("drugcharacterization") == "1" and d.get("medicinalproduct")
        ]
        return len(suspect) == 1 and bool(reactions) and bool(reactions[0].get("reactionmeddrapt"))
    except (AttributeError, TypeError, IndexError):
        return False


broken_inputs = [
    ("empty dict", {}),
    ("None", None),
    ("drug field is a string, not a list", {"patient": {"drug": "oops", "reaction": [{"reactionmeddrapt": "X"}]}}),
    ("missing patient entirely", {"foo": "bar"}),
]
for label, bad in broken_inputs:
    print(f"{label}: {is_usable_record(bad)}")

real_usable = next(r for r in all_records if flatten_record(r) is not None)
print("a real, genuinely usable record:", is_usable_record(real_usable))

server.shutdown()


empty dict: False
None: False
drug field is a string, not a list: False
missing patient entirely: False
a real, genuinely usable record: True


---
### Grading notes

- Exercise 2/3's record count (300) and last `safety_report_id` (`10003602`) depend
  only on the practice sample file being complete and untouched -- if a student's
  count is off, check whether `openfda_practice_sample.json` was accidentally
  modified or partially downloaded.
- Exercise 3's retry line only appears once per server run -- the practice server
  rate-limits `skip=200` exactly once, then remembers it's already done that. A
  student re-running the cell without restarting the server won't see the message
  fire again, which is correct behavior, not a bug.
- Exercise 4's 191/300 usable count is real, not a round number -- real reports are
  messier than the already-cleaned files from Sessions 1.1-1.2 (191 is consistent with
  the ~60-65% usable rate this project has seen at every scale so far). If a student's
  count differs, check whether they're requiring exactly one suspect drug, or
  accepting "at least one."
- Exercise 5 must return `False`, never raise, on every broken input -- a student
  who only catches `KeyError` will crash on the "drug field is a string" case
  specifically, since that raises `AttributeError` (calling `.get()` on a string
  character) or `TypeError`, not `KeyError`.
